# Capstone companion --- Chapter 11: Failure Modes, Adversarial Testing and Design of Experiments

Chapter~11 develops a discipline for locating an agent's failures rather than merely counting them. A factor-balanced suite varies the inputs along named axes, the agent is run across the resulting cells, and each failure is attributed to a factor level by a logistic model whose deviance test asks whether that factor explains the failures beyond chance. This companion reads that construction on the capstone banking complaint agent, using the campaign artifact `data/capstone_run.json` produced by the testing harness in `agentlab/testing`.

The suite crosses three input factors. `clarity` controls whether the customer message states its issue plainly, ambiguously, or misleadingly; `entity_aliasing` controls whether products and parties are named canonically, by alias, or with a typo; and `reasoning_cue` controls whether the prompt carries a chain-of-thought cue, a misleading cue, or none. Twenty seed cases are each realized across the factor levels, so the campaign is a balanced design over the input variations rather than an ad hoc collection of examples.

In [ ]:
import json
from pathlib import Path

root = Path('.') if Path('data').exists() else Path('..')
run = json.loads((root / 'data' / 'capstone_run.json').read_text())
print('runs in campaign :', run['n_runs'])
print('workflow adherence:', run['workflow_adherence'])
print('audit verifies    :', run['audit_verifies'])
print('overall accuracy  :', run['overall']['accuracy'])
print('accuracy meaning  :', run['overall']['definition'])

## The suite is balanced

Attribution is only interpretable when the design is balanced, so that a factor level is not confounded with the difficulty of the seed cases it happens to appear in. The artifact records the realized counts per seed and per clarity level; the twenty seeds each appear the same number of times, and the clarity levels are close to even.

In [ ]:
seeds = run['seed_balance']
print('distinct seeds     :', len(seeds))
print('runs per seed (set):', sorted(set(seeds.values())))
print('clarity balance    :', run['clarity_balance'])

## Factor attribution over the whole campaign

For each factor, a logistic model regresses the per-run failure indicator on the factor levels. The likelihood-ratio statistic $G^2$ measures how much deviance the factor removes, and its $p$-value is adjusted for multiple factors. The decision rule of Chapter~11 is that a factor is implicated only when its adjusted $p$-value falls below the chosen level. The table below reports the real values from the campaign.

In [ ]:
print(f"{'factor':16s} {'G2':>7s} {'p_adj':>8s} {'pseudo_r2':>10s} {'significant':>12s}")
print('-' * 56)
for row in run['overall']['attribution']:
    print(f"{row['factor']:16s} {row['G2']:7.2f} {row['p_adj']:8.4f} "
          f"{row['pseudo_r2']:10.3f} {str(row['significant_adj']):>12s}")

No factor is significant after correction. The smallest adjusted $p$-value belongs to `clarity` at roughly $0.96$, far above any conventional level, and its pseudo-$R^2$ of about $0.015$ shows that the factor explains almost none of the variation in failure. The reading follows Chapter~11 exactly: the campaign does not license a claim that any single input axis drives the agent's failures. Clarity is merely the nearest of the three, not an implicated cause.

In [ ]:
clarity = next(r for r in run['overall']['attribution'] if r['factor'] == 'clarity')
print('nearest factor    :', clarity['factor'])
print('adjusted p-value  :', clarity['p_adj'])
print('significant_adj   :', clarity['significant_adj'])
print('level odds ratios :', clarity['odds_ratios'])
assert not any(r['significant_adj'] for r in run['overall']['attribution']), \
    'no whole-campaign factor should be significant'

## Decomposing the weak link by component

A campaign-wide null result does not mean the agent is uniformly healthy. Chapter~11 recommends decomposing failures by the component that produced them, because a factor that is invisible in the aggregate may concentrate in one tool. The artifact records the weak-link count per component: the number of runs in which each tool was the failing step.

In [ ]:
weak = run['weak_link']
for comp, count in sorted(weak.items(), key=lambda kv: -kv[1]):
    print(f'{comp:20s} failing-step count = {count}')
print()
print('weak link concentrates in:', max(weak, key=weak.get))

The failures concentrate in `classify_complaint` (nine failing runs) with a single failing run in `flag_regulatory`. Attribution is therefore most informative when restricted to the component that carries the failures. The per-component deviance table below repeats the logistic attribution within each tool that had enough failures to fit.

In [ ]:
abc = run['attribution_by_component']
for comp, block in abc.items():
    print(f"=== {comp}  (scored={block['scored']}, failures={block['failures']}) ===")
    dev = block.get('deviance')
    if not dev:
        print('  too few failures to fit a per-factor model')
        continue
    for factor, d in sorted(dev.items(), key=lambda kv: kv[1]['p_adj']):
        print(f"  {factor:16s} G2={d['G2']:6.2f}  p_adj={d['p_adj']:.4f}  "
              f"pseudo_r2={d['pseudo_r2']:.3f}  sig={d['significant_adj']}")

Within `classify_complaint`, clarity is again the nearest factor, with an adjusted $p$-value around $0.17$ and the highest pseudo-$R^2$ of the block, yet it still does not clear the significance threshold. The per-level failure rates make the tendency concrete without overstating it: the ambiguous level fails more often than the clear level, but the sample is too small for the deviance test to rule out chance.

In [ ]:
clf = abc['classify_complaint']['by_factor']['clarity']
print(f"{'clarity level':16s} {'failed':>7s} {'n':>4s} {'rate':>7s}")
for level, cell in clf.items():
    print(f"{level:16s} {cell['failed']:7d} {cell['n']:4d} {cell['rate']:7.3f}")

## The harness that produced the campaign

The artifact is the output of the capstone testing harness, which runs the balanced suite against the agent and fits the attribution models. The companion reads the pinned result rather than re-running the campaign, but the harness class can be imported to confirm the provenance of the numbers above.

In [ ]:
from agentlab.testing.capstone_harness import CapstoneTestHarness
print('harness class     :', CapstoneTestHarness.__name__)
print('defined in module :', CapstoneTestHarness.__module__)

This is the capstone's realization of Chapter~11. The banking agent is exercised by a factor-balanced suite, each failure is attributed to a factor level by a logistic deviance test, and the campaign reports a null result: no input factor is implicated after correction, with clarity the nearest at an adjusted $p$-value near $0.96$. Decomposing by component then localizes the failures to `classify_complaint`, which is where subsequent work would be directed. Chapter~16 assembles this attribution logic into the governed testing workflow that decides whether the agent is fit to ship.